# Best Camera Model, Round 4: HPO at the Car's Timing

Rounds 1–3 decided once every 20 ms (50 Hz). At that rate a 4-frame stack
spans 60 ms, about 4 cm of travel at 2 m/s, so the frames are nearly
identical. When rounds 1 and 2 searched `frame_stack ∈ {1, 4}` and picked 1,
stacking never got a fair test. The DeepRacer Evo decides at its camera rate
(~15 fps). Genesis ADR 0005 (`dev` cb7b0c9) adds the timing this round fixes:

- **Decision period** `control_dt_s = 0.07` s, a **placeholder**
  (`CONTROL_DT_S`, next cell) until the Evo measurements arrive.
  `frame_interval_s` equals it, so the stack holds one frame per decision and
  a 4-frame stack spans 210 ms.
- **Actuation-timing DR, on in every trial**: per-episode command latency
  30–90 ms, ±10 ms jitter per decision, and 5 % dropped decisions. These are
  the catalog placeholders. The spec takes one drop probability, so 5 % is
  the mean of the catalog's 0–0.1 range.
- **Discount horizon in seconds**: `horizon_s` replaces `gamma`
  (`gamma = exp(−T/H)`), so the searched value means the same thing at any
  decision rate.

**Searched**: round 2's space (`lr`, `entropy_coef`, `epochs`, `minibatches`,
`frame_stack`, `dr`), with `horizon_s` in place of `gamma`. **Fixed**: round
3's winning reward (`speed_target` at 1.6 m/s) and round 3's completion-first
score. **Budget**: round 2's, 14 trials × 2.4 M env-steps. The budget unit is
trials × step cap, not GPU hours, so each trial covers 3.5× more simulated
driving than at 50 Hz. The first two trials are round 2's winner at
`frame_stack` 1 and 4, a paired comparison before TPE takes over.

The study is built to run on a Colab G4 (see *Running on Colab G4* at the
end). Records go to `runs/best_camera_car_timing/` (`study.db`,
`logs/*.result.json`, `report.csv`). The helpers cell pins the CWD to the
repo root, so a per-notebook CWD can't resolve `runs/` to `experiments/runs/`.

In [ ]:
# --- configuration (papermill parameters cell) ------------------------------
SEED = 0
ROOT = "runs/best_camera_car_timing"
REUSE = False  # replay recorded step results

# The car's decision period. PLACEHOLDER until the DeepRacer Evo measurements
# arrive (Sandha et al.'s 62-71 ms frame window; sim.dt 0.01 x 7).
CONTROL_DT_S = 0.07
LATENCY_MS = (30.0, 90.0)  # per-episode base command latency (catalog)
LATENCY_JITTER_MS = 10.0  # +/- per decision (catalog)
DECISION_DROP = 0.05  # per decision; mean of the catalog's 0-0.1

TRAIN_TRACKS = (
    "reinvent_base",
    "Oval_track",
    "Mexico_track",
    "China_track",
    "reInvent2019_track",
    "2022_summit_speedway",
    "caecer_loop",
)
NUM_ENVS = 128
RESOLUTION = (160, 120)
BACKEND = "gpu"  # "cpu" = rasterizer, for smoke runs only

# round-1 HPO winner, carried through rounds 2-3 (round 2's anchor adds dr=True)
BEST = {
    "lr": 1.5529021289627495e-4,
    "entropy_coef": 3.68356162276608e-3,
    "epochs": 5,
    "minibatches": 4,
    "gamma": 0.995,
}

# round-3 winning reward (runs/best_camera_completion/study.db, trial 2)
TARGET_SPEED = 1.6
REWARD_SCALES = {
    "progress": 10.0,
    "speed": 0.6,
    "centered": 1.0,
    "heading": 0.5,
    "steering": 0.3,
    "action_rate": 0.05,
    "off_track": 2.0,
}

# HPO budget = round 2's (best_camera_diverse): 14 trials x 2.4M env-steps
HPO_TRIALS = 14
HPO_DEADLINE_H = 12.0  # safety stop; rerunning with REUSE resumes
TRIAL_STEPS = 2_400_000
TRIAL_EVAL_EVERY = 1_200_000
BENCH_STEPS = 150

In [ ]:
"""Session helpers: CWD pin, result caching, figure display, budget clock."""

import io
import json
import os
import time
from pathlib import Path

from deepracer_genesis.experiment import run
from deepracer_genesis.experiment.builder import Builder
from deepracer_genesis.experiment.evaluator import evaluate_policy
from IPython.display import Image, display


def studies_root() -> Path:
    """Return the deepracer-studies checkout that holds the current directory.

    Raises:
        FileNotFoundError: If no parent directory is the studies checkout.
    """
    cwd = Path.cwd().resolve()
    for folder in (cwd, *cwd.parents):
        pyproject = folder / "pyproject.toml"
        if (
            pyproject.is_file()
            and (folder / "experiments").is_dir()
            and 'name = "deepracer-studies"' in pyproject.read_text()
        ):
            return folder
    raise FileNotFoundError(f"{cwd} is not inside the deepracer-studies checkout")


# ROOT is repo-root relative. A per-notebook CWD (JupyterLab, or papermill
# launched from experiments/) would resolve it to experiments/runs/, miss every
# record and retrain silently, so pin the CWD before anything touches the disk.
os.chdir(studies_root())
print(f"cwd={os.getcwd()}")

# survive a shared GPU: cap Madrona's device heap (its default does not fit
# next to a resident LLM server) and let torch grow segments on demand
os.environ.setdefault("MADRONA_MWGPU_DEVICE_HEAP_SIZE", str(1 << 30))
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

T0 = time.time()
LOGS = os.path.join(ROOT, "logs")
os.makedirs(LOGS, exist_ok=True)


def elapsed_h() -> float:
    """Wall-clock hours since the notebook started."""
    return (time.time() - T0) / 3600.0


def recorded(name: str) -> dict | None:
    """Return a step's recorded result when REUSE is on and it exists."""
    path = os.path.join(LOGS, f"{name}.result.json")
    if REUSE and os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None


def record_result(name: str, result: dict) -> dict:
    """Write a step's result JSON and return it."""
    with open(os.path.join(LOGS, f"{name}.result.json"), "w") as f:
        json.dump(result, f)
    return result


def cached_run(name: str, spec, on_eval=None) -> dict:
    """Train ``spec`` in-process, or return the step's recorded result.

    Args:
        name: Step name; keys the result JSON under ``LOGS``.
        spec: The built ExperimentSpec to train.
        on_eval: Optional periodic-eval callback; raising inside it stops the run.

    Returns:
        ``{"metrics", "history", "holdout", "train", "run_dir"}``.
    """
    result = recorded(name)
    if result is not None:
        return result
    rec = run(spec, root=ROOT, on_eval=on_eval)
    return record_result(
        name,
        {
            "metrics": rec.metrics,
            "history": rec.eval_history,
            "holdout": rec.holdout,
            "train": rec.train,
            "run_dir": spec.run_dir(ROOT),
        },
    )


def show_fig(fig) -> None:
    """Render a figure inline whatever the active matplotlib backend."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    display(Image(buf.getvalue()))


def bench(name: str, spec, steps: int = BENCH_STEPS) -> dict:
    """Measure a spec's rollout throughput with a constant actor (cached by name).

    Args:
        name: Step name for the result cache.
        spec: The built spec to bench.
        steps: Timed control steps.

    Returns:
        ``{"env_steps_per_s": float}``.
    """
    result = recorded(name)
    if result is not None:
        return result
    import torch

    sim = Builder(spec).sim()
    action = torch.zeros(sim.num_envs, 2, device=sim.device)
    action[:, 1] = 0.4

    def actor(td):
        td.set("action", action)
        return td

    with torch.inference_mode():
        evaluate_policy(sim, actor, steps=min(30, steps))  # warmup / JIT
        t0 = time.perf_counter()
        evaluate_policy(sim, actor, steps=steps)
        dt = time.perf_counter() - t0
    return record_result(name, {"env_steps_per_s": round(steps * sim.num_envs / dt, 1)})

In [ ]:
"""The reward: round 3's winner, the speed term pulled toward a target pace."""

from deepracer_genesis.envs.rewards import deepracer, reads, reward_reads


# it reads exactly the built-in's signals, so genesis's K.5 check can verify it
@reads(*reward_reads(deepracer))
def reward_speed_target(env):
    """Built-in shaping with ``speed`` replaced by ``-|v - target_speed| * dt``.

    ``target_speed`` comes from ``env.reward_params``, so it is in the spec hash.

    Args:
        env: The live DeepRacerEnv.

    Returns:
        The built-in term dict with the ``speed`` term rewritten.
    """
    terms = deepracer(env)
    target = env.reward_params["target_speed"]
    terms["speed"] = -(env.v_forward - target).abs() * env.dt
    return terms

In [ ]:
"""Spec building: the car's timing fixed, the trial's hyperparameters injected."""

import math

import matplotlib.pyplot as plt
import pandas as pd
from deepracer_genesis.experiment import (
    PPO,
    AsymmetricCameraPolicy,
    CameraEnvironment,
    DomainRandomizationActions,
    DomainRandomizationCamera,
    DomainRandomizationPhysics,
    DomainRandomizationTrackAppearance,
    RewardShaping,
)
from deepracer_genesis.experiment.rsl_backend import discount_for_horizon

# papermill injects overrides after the parameters cell, so derive from them here
TRAIN_TRACKS = tuple(TRAIN_TRACKS)
RESOLUTION = tuple(RESOLUTION)
LATENCY_MS = tuple(LATENCY_MS)
FRAME_INTERVAL_S = CONTROL_DT_S  # one stacked frame per decision, as on the car

# Horizons in seconds (ADR 0005: gamma = exp(-T / H)). Rounds 1-3 ran at
# T = 0.02 s with PPO's default gae_lambda 0.95: lambda keeps that time horizon
# (0.39 s, lambda 0.836 at 70 ms) and is not searched, as in round 2.
DT_50HZ_S = 0.02
LAM_HORIZON_S = -DT_50HZ_S / math.log(0.95)
BEST_HORIZON_S = -DT_50HZ_S / math.log(BEST["gamma"])  # 3.99 s
HORIZON_RANGE_S = (0.5, 8.0)

# round 2's winner (round-1 config + dr=True), with its horizon in seconds
ANCHOR = {
    "lr": BEST["lr"],
    "entropy_coef": BEST["entropy_coef"],
    "epochs": BEST["epochs"],
    "minibatches": BEST["minibatches"],
    "horizon_s": BEST_HORIZON_S,
    "dr": True,
}


def car_timing_spec(
    *,
    frame_stack: int,
    horizon_s: float,
    lr: float,
    entropy_coef: float,
    epochs: int,
    minibatches: int,
    dr: bool = True,
    steps: int = TRIAL_STEPS,
    eval_every: int = TRIAL_EVAL_EVERY,
    variant: str = "probe",
):
    """Build one camera spec at the car's timing; actuation-timing DR is always on.

    ``dr`` toggles round 2's appearance/camera/physics stack and command noise.

    Args:
        frame_stack: Frames stacked along channels, ``FRAME_INTERVAL_S`` apart.
        horizon_s: Discount horizon in seconds (sets ``gamma``).
        lr: PPO learning rate.
        entropy_coef: PPO entropy bonus.
        epochs: PPO epochs per iteration.
        minibatches: PPO minibatches per epoch.
        dr: Apply the non-timing DR stack.
        steps: Total env-steps (policy decisions x envs).
        eval_every: Periodic-eval cadence.
        variant: Run-dir variant name.

    Returns:
        The validated frozen spec.
    """
    pipe = CameraEnvironment(
        render="madrona",
        backend=BACKEND,
        resolution=RESOLUTION,
        num_envs=NUM_ENVS,
        tracks=TRAIN_TRACKS,
        frame_stack=frame_stack,
        random_direction=True,
        control_dt_s=CONTROL_DT_S,
        # a single frame has no spacing, and genesis refuses the field there
        frame_interval_s=FRAME_INTERVAL_S if frame_stack >= 2 else None,
    )
    if dr:
        pipe = (
            pipe
            >> DomainRandomizationTrackAppearance(strength=0.6)
            >> DomainRandomizationCamera(
                brightness=(0.7, 1.3), hue=0.05, blur=0.3, camera_jitter=True
            )
            >> DomainRandomizationPhysics()
        )
    pipe = pipe >> AsymmetricCameraPolicy(
        actor_keys=("camera",),
        critic_keys=("camera", "state"),
        distribution={"std_range": (0.1, 1.0)},
    )
    # latency_ms replaces rounds 1-3's delay_steps=1 (genesis refuses both)
    pipe = pipe >> DomainRandomizationActions(
        steer_noise=0.02 if dr else 0.0,
        speed_noise=0.05 if dr else 0.0,
        latency_ms=LATENCY_MS,
        latency_jitter_ms=LATENCY_JITTER_MS,
        decision_drop=DECISION_DROP,
    )
    pipe = pipe >> RewardShaping(
        fn=reward_speed_target,
        scales=REWARD_SCALES,
        params={"target_speed": TARGET_SPEED},
    )
    # PPO's rollout length stays at its default 24 decisions (1.68 s at 70 ms)
    pipe = pipe >> PPO(
        lr=lr,
        entropy_coef=entropy_coef,
        epochs=epochs,
        minibatches=minibatches,
        horizon_s=horizon_s,
        lam_horizon_s=LAM_HORIZON_S,
    )
    return pipe.build(
        seed=SEED,
        total_env_steps=steps,
        eval_every_steps=eval_every,
        group="best_camera_car_timing_hpo",
        variant=variant,
    )


print(
    f"root={ROOT} control_dt={CONTROL_DT_S * 1000:.0f} ms "
    f"lam_horizon={LAM_HORIZON_S:.2f} s anchor_horizon={BEST_HORIZON_S:.2f} s "
    f"reuse={REUSE}"
)

## Preflight

Every corner of the search space must build and validate before any GPU time
is spent. Then one rollout bench at 70 ms projects the HPO's hours from round
2's measured ratio of training rate to rollout rate. The projection excludes
scene builds and evals; the trial loop prints a measured ETA after each trial.

In [ ]:
"""Preflight: the search space's corners build; the rollout bench projects cost."""

from itertools import product

for fs, h, use_dr in product((1, 4), HORIZON_RANGE_S, (True, False)):
    corner = car_timing_spec(**{**ANCHOR, "horizon_s": h, "dr": use_dr}, frame_stack=fs)
    print(
        f"fs={fs} horizon_s={h:<4} dr={use_dr!s:<5} id={corner.id()} "
        f"gamma={discount_for_horizon(h, CONTROL_DT_S):.4f}"
    )

# round 2 on the 4060 Ti: trials trained at 2,517 env-steps/s against an
# 8,422 env-steps/s rollout bench, so training ran at ~0.30x the rollout rate
TRAIN_PER_ROLLOUT = 2517 / 8422

r = bench("bench_fs4", car_timing_spec(**ANCHOR, frame_stack=4))
train_sps = r["env_steps_per_s"] * TRAIN_PER_ROLLOUT
print(
    f"rollout {r['env_steps_per_s']:,.0f} env-steps/s at "
    f"{CONTROL_DT_S * 1000:.0f} ms -> HPO ~ "
    f"{HPO_TRIALS * TRIAL_STEPS / train_sps / 3600:.1f} h of training"
)

## HPO at the car's timing

Same score as round 3, `completion_rate × 100 + 0.05 × mean_progress_m`.
Completion ranks first, and progress breaks ties while nothing finishes laps.
When completion is 0 everywhere this reduces to round 2's progress ranking.
There are no takeoff redraws, which keeps the budget equal to round 2's.

`horizon_s` is searched log-uniform over 0.5–8 s. The 50 Hz `gamma` choices
{0.99, 0.995} are horizons of 2 s and 4 s, and the 0.995 winner is 4 s, so
the range keeps both inside it, with 2 s at its geometric centre. 0.5 s
(7 decisions, `gamma` 0.87) is roughly the shortest look-ahead that still
reaches the corner a steering command sets up: 0.8 m at the 1.6 m/s target.
8 s (`gamma` 0.991) stays under one lap of the shortest training circuit
(`reinvent_base`, 17.7 m, 11 s at the target). `entropy_coef` keeps round 2's
range. Per-step rewards are 3.5× larger at 70 ms (the terms are × dt), so the
effective entropy weight drops ~3.5×, well inside the range's 67× span.

In [ ]:
from optuna import Trial, create_study
from optuna.logging import WARNING, set_verbosity
from optuna.samplers import TPESampler

set_verbosity(WARNING)


def suggest(trial: Trial) -> dict:
    """Sample round 2's search space, with ``horizon_s`` in place of ``gamma``."""
    return {
        "lr": trial.suggest_float("lr", 3e-5, 6e-4, log=True),
        "entropy_coef": trial.suggest_float("entropy_coef", 3e-4, 2e-2, log=True),
        "epochs": trial.suggest_int("epochs", 3, 6),
        "minibatches": trial.suggest_categorical("minibatches", [4, 8]),
        "frame_stack": trial.suggest_categorical("frame_stack", [1, 4]),
        "horizon_s": trial.suggest_float("horizon_s", *HORIZON_RANGE_S, log=True),
        "dr": trial.suggest_categorical("dr", [True, False]),
    }


def objective(trial: Trial) -> float:
    """Train one config at the car's timing and score it completion-first.

    Args:
        trial: The Optuna trial supplying hyperparameters.

    Returns:
        ``completion_rate * 100 + 0.05 * mean_progress_m`` of the final eval.
    """
    params = suggest(trial)
    t0 = time.time()
    spec = car_timing_spec(variant=f"t{trial.number:03d}", **params)
    m = cached_run(f"trial_{trial.number:03d}", spec)["metrics"]
    trial.set_user_attr("completion", m["completion_rate"])
    trial.set_user_attr("progress_m", m["mean_progress_m"])
    trial.set_user_attr(
        "gamma", discount_for_horizon(params["horizon_s"], CONTROL_DT_S)
    )
    trial.set_user_attr("wall_min", (time.time() - t0) / 60)
    print(
        f"  trial {trial.number} [fs={params['frame_stack']} "
        f"H={params['horizon_s']:.2f}s dr={params['dr']}]: "
        f"completion={m['completion_rate']:.3f} "
        f"progress={m['mean_progress_m']:.2f}m"
    )
    return float(m["completion_rate"] * 100 + 0.05 * m["mean_progress_m"])


def finished(study) -> int:
    """Count the study's finished trials (complete, pruned or failed)."""
    return sum(t.state.is_finished() for t in study.trials)


study = create_study(
    direction="maximize",
    study_name="best_camera_car_timing",
    sampler=TPESampler(seed=SEED),
    storage=f"sqlite:///{ROOT}/study.db",
    load_if_exists=True,
)
if not study.trials:  # anchors: round 2's winner at both depths
    for frame_stack in (1, 4):
        study.enqueue_trial({**ANCHOR, "frame_stack": frame_stack})

while finished(study) < HPO_TRIALS and elapsed_h() < HPO_DEADLINE_H:
    study.optimize(objective, n_trials=1)
    walls = [
        t.user_attrs["wall_min"] for t in study.trials if "wall_min" in t.user_attrs
    ]
    left = HPO_TRIALS - finished(study)
    if walls and left > 0:
        eta_h = left * sum(walls) / len(walls) / 60
        print(f"  {left} trials left, ETA ~ {eta_h:.1f} h")

## Frame stack at the car's timing

The question this round exists for: with frames 70 ms apart, does a 4-frame
stack beat a single frame? Read the anchor pair first (same config, only the
stack differs), then each arm's best and median across the study.

In [ ]:
"""Both stack depths against the discount horizon, plus the recorded table."""

trials = pd.DataFrame(
    [
        {
            "trial": t.number,
            "state": t.state.name,
            "objective": t.value,
            "completion": t.user_attrs.get("completion"),
            "progress_m": t.user_attrs.get("progress_m"),
            "gamma": t.user_attrs.get("gamma"),
            **t.params,
        }
        for t in study.trials
    ]
).sort_values("objective", ascending=False)
display(trials)
trials.to_csv(os.path.join(ROOT, "report.csv"), index=False)

done = trials[trials["state"] == "COMPLETE"]
display(
    done.groupby("frame_stack")[["objective", "completion", "progress_m"]]
    .agg(["count", "max", "median"])
    .round(3)
)

# categorical slots 1-2 of the reference palette; shape repeats the identity
ARMS = ((1, "#2a78d6", "o"), (4, "#eb6834", "s"))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (col, label) in zip(
    axes, (("progress_m", "mean progress (m)"), ("completion", "completion rate"))
):
    for fs, color, marker in ARMS:
        arm = done[done["frame_stack"] == fs]
        ax.scatter(
            arm["horizon_s"],
            arm[col],
            s=64,
            color=color,
            marker=marker,
            edgecolors="white",
            linewidths=2,
            label=f"frame_stack={fs}",
        )
    ax.set(xscale="log", xlabel="horizon_s (s, log scale)", ylabel=label)
    ax.set_xlim(HORIZON_RANGE_S[0] / 1.1, HORIZON_RANGE_S[1] * 1.1)  # searched
    ax.grid(alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
axes[1].set_ylim(-0.03, 1.03)  # a rate
axes[0].legend(frameon=False)
fig.suptitle(
    f"HPO at {CONTROL_DT_S * 1000:.0f} ms decisions: "
    "frame stack against discount horizon"
)
fig.tight_layout()
show_fig(fig)

pair = done[done["trial"] < 2].set_index("frame_stack")
print(
    "anchor pair (round 2 winner, horizon 3.99 s):",
    {int(fs): round(v, 3) for fs, v in pair["objective"].items()},
)
print("winner:", study.best_params, "->", round(study.best_value, 3))

## Running on Colab G4

The VM installs from the committed `uv.lock` (Python 3.12 under uv; the VM's
own Python is 3.13). Madrona needs the EGL Vulkan ICD. Recipe:
`deepracer-genesis/docs/usage/google_colab_cli.md`. From the repo root:

```bash
tar czf /tmp/car_timing.tgz pyproject.toml uv.lock experiments/__init__.py \
    experiments/best_camera_car_timing.ipynb    # + runs/best_camera_car_timing to resume
colab new -s cartiming --gpu G4
colab upload /tmp/car_timing.tgz /content/car_timing.tgz -s cartiming
colab exec -s cartiming -f setup_g4.py --timeout 1800
colab exec -s cartiming -f launch.py              # returns at once; the VM keeps working
```

`setup_g4.py`:

```python
import json, os, subprocess
os.makedirs("/etc/vulkan/icd.d", exist_ok=True)
with open("/etc/vulkan/icd.d/nvidia_icd.json", "w") as f:
    json.dump({"file_format_version": "1.0.0",
               "ICD": {"library_path": "libEGL_nvidia.so.0", "api_version": "1.3.0"}}, f)
subprocess.run(
    "mkdir -p /content/studies && tar xzf /content/car_timing.tgz -C /content/studies"
    " && pip uninstall -q -y tensorflow tensorflow-cpu tf-keras && pip install -q uv"
    " && cd /content/studies && uv python install 3.12"
    " && uv sync --frozen --no-install-project -p 3.12"
    " && .venv/bin/python -m ipykernel install --user --name deepracer-studies",
    shell=True, check=True)
```

`launch.py` (papermill with the venv's kernel; `REUSE True` makes a relaunch
resume from `study.db` and `logs/`):

```python
import subprocess
subprocess.Popen(
    "cd /content/studies && mkdir -p runs/best_camera_car_timing && setsid nohup"
    " .venv/bin/papermill -k deepracer-studies --log-output -p REUSE True"
    " experiments/best_camera_car_timing.ipynb"
    " runs/best_camera_car_timing/executed.ipynb"
    " > runs/best_camera_car_timing/papermill.log 2>&1 &", shell=True)
```

Poll by running a one-line `exec` file that prints the tail of
`/content/studies/runs/best_camera_car_timing/papermill.log`. When it ends,
`tar czf /content/records.tgz` the study's `study.db`, `logs/`, `report.csv`
and `executed.ipynb` under `runs/best_camera_car_timing/`.
Then `colab download /content/records.tgz records.tgz -s cartiming`, unpack
it at the repo root, move `executed.ipynb` over this notebook (outputs are
the research record), and `colab stop -s cartiming`.

**Expected cost**: round 2 ran these 14 × 2.4 M steps in 4.1 h on the 4060 Ti
(2,517 env-steps/s). At 70 ms each step runs 7 physics substeps instead of 2,
with the same single render. If physics were the whole rollout, the step
would cost 1.75× as much, so the same card would need 4.1–7.1 h. Plan for up
to ~7 G4 hours plus ~20 min of setup. The preflight cell prints a measured
projection within minutes; stop the session if it is far off.

## Notes

- **Placeholders**: `CONTROL_DT_S`, the latency range, jitter and drop rate
  all wait on the Evo measurements. Changing any of them changes every spec
  id, so the study must be rerun under a new `ROOT`.
- **No final run**: this round answers `frame_stack` and `horizon_s` at the
  car's timing. Scale the winner with round 3's takeoff-guarded final cell
  once the answer is in.
- **Reproducibility**: seeded end to end. The study resumes from
  `runs/best_camera_car_timing/study.db`, step records are in
  `runs/best_camera_car_timing/logs/`, and `REUSE = True` replays them.